In [ ]:
import json
import pandas as pd

with open('../data/mitre/enterprise-attack.json', 'r') as f:
    data = json.load(f)

# Extraire techniques ET sous-techniques
techniques = [obj for obj in data['objects'] if obj['type'] == 'attack-pattern']

records = []
for t in techniques:
    ext_refs = t.get('external_references', [{}])
    technique_id = ext_refs[0].get('external_id', '')
    
    # Identifier si c'est une sous-technique (contient un point ex: T1110.001)
    is_subtechnique = '.' in technique_id
    parent_id = technique_id.split('.')[0] if is_subtechnique else None
    
    tactic = [p['phase_name'] for p in t.get('kill_chain_phases', [])]
    
    records.append({
        'technique_id':    technique_id,
        'name':            t.get('name', ''),
        'is_subtechnique': is_subtechnique,
        'parent_id':       parent_id,
        'tactics':         tactic,
        'description':     t.get('description', '')[:150]
    })

df = pd.DataFrame(records)
print(f"Total techniques     : {len(df[~df['is_subtechnique']])}")
print(f"Total sous-techniques: {len(df[df['is_subtechnique']])}")
print("\nExemple sous-techniques Brute Force :")
print(df[df['parent_id'] == 'T1110'][['technique_id','name','tactics']])

In [ ]:
# Mapping complet : label → technique + sous-techniques
extended_mapping = {
    "NORMAL": {
        "att_ck_technique_id":   None,
        "att_ck_technique_name": None,
        "att_ck_tactic":         None,
        "subtechniques":         [],
        "criticality_score":     0,
        "color":                 "#00ff88"
    },
    "BRUTE_FORCE": {
        "att_ck_technique_id":   "T1110",
        "att_ck_technique_name": "Brute Force",
        "att_ck_tactic":         "Credential Access",
        "subtechniques": [
            {"id": "T1110.001", "name": "Password Guessing"},
            {"id": "T1110.002", "name": "Password Cracking"},
            {"id": "T1110.003", "name": "Password Spraying"},
            {"id": "T1110.004", "name": "Credential Stuffing"}
        ],
        "criticality_score": 6,
        "color": "#ffd600"
    },
    "PORT_SCAN": {
        "att_ck_technique_id":   "T1046",
        "att_ck_technique_name": "Network Service Discovery",
        "att_ck_tactic":         "Discovery",
        "subtechniques": [],
        "criticality_score": 5,
        "color": "#00d4ff"
    },
    "DoS": {
        "att_ck_technique_id":   "T1499",
        "att_ck_technique_name": "Endpoint Denial of Service",
        "att_ck_tactic":         "Impact",
        "subtechniques": [
            {"id": "T1499.001", "name": "OS Exhaustion Flood"},
            {"id": "T1499.002", "name": "Service Exhaustion Flood"},
            {"id": "T1499.003", "name": "Application Exhaustion Flood"}
        ],
        "criticality_score": 8,
        "color": "#ff8800"
    },
    "DDoS": {
        "att_ck_technique_id":   "T1498",
        "att_ck_technique_name": "Network Denial of Service",
        "att_ck_tactic":         "Impact",
        "subtechniques": [
            {"id": "T1498.001", "name": "Direct Network Flood"},
            {"id": "T1498.002", "name": "Reflection Amplification"}
        ],
        "criticality_score": 9,
        "color": "#ff4400"
    },
    "WEB_ATTACK": {
        "att_ck_technique_id":   "T1190",
        "att_ck_technique_name": "Exploit Public-Facing Application",
        "att_ck_tactic":         "Initial Access",
        "subtechniques": [],
        "criticality_score": 7,
        "color": "#ff3366"
    },
    "BOT": {
        "att_ck_technique_id":   "T1071",
        "att_ck_technique_name": "Application Layer Protocol",
        "att_ck_tactic":         "Command and Control",
        "subtechniques": [
            {"id": "T1071.001", "name": "Web Protocols"},
            {"id": "T1071.002", "name": "File Transfer Protocols"},
            {"id": "T1071.004", "name": "DNS"}
        ],
        "criticality_score": 10,
        "color": "#9d4edd"
    }
}

# Sauvegarder
with open('../data/attck_extended_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(extended_mapping, f, indent=2, ensure_ascii=False)

print("✅ Mapping étendu sauvegardé !")
for label, info in extended_mapping.items():
    nb_sub = len(info['subtechniques'])
    print(f"  {label:15} → {info['att_ck_technique_id']} ({nb_sub} sous-techniques)")

In [ ]:
# Écrire le nouveau attck_mapper.py directement depuis le notebook
new_mapper = '''import json
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

with open(os.path.join(BASE_DIR, 'data', 'attck_extended_mapping.json'), 'r') as f:
    LABEL_TO_ATTCK = json.load(f)

def map_prediction_to_attck(label_name: str, confidence: float,
                             features: dict = None) -> dict:
    """
    Mappe une prédiction ML vers ATT&CK complet.
    Retourne le format event_contract.json pour P4.
    """
    info = LABEL_TO_ATTCK.get(label_name, {
        'att_ck_technique_id':   None,
        'att_ck_technique_name': None,
        'att_ck_tactic':         None,
        'subtechniques':         [],
        'criticality_score':     0,
        'color':                 '#888888'
    })

    # Choisir la sous-technique la plus probable selon les features
    best_subtechnique = _select_subtechnique(label_name, info['subtechniques'], features)

    return {
        'predicted_label':         label_name,
        'confidence_score':        round(confidence, 4),
        'att_ck_technique_id':     info['att_ck_technique_id'],
        'att_ck_technique_name':   info['att_ck_technique_name'],
        'att_ck_tactic':           info['att_ck_tactic'],
        'subtechnique_id':         best_subtechnique.get('id') if best_subtechnique else None,
        'subtechnique_name':       best_subtechnique.get('name') if best_subtechnique else None,
        'all_subtechniques':       info['subtechniques'],
        'criticality_score':       info['criticality_score'],
        'color':                   info['color'],
        'alert_level':             _get_alert_level(info['criticality_score'])
    }

def _select_subtechnique(label: str, subtechniques: list, features: dict) -> dict:
    """Logique simple pour choisir la sous-technique selon les features."""
    if not subtechniques:
        return None
    if features is None:
        return subtechniques[0]

    # Brute Force : choisir selon le nombre de tentatives
    if label == 'BRUTE_FORCE':
        failed = features.get('num_failed_logins', 0)
        if failed > 100:  return subtechniques[2]  # Password Spraying
        if failed > 50:   return subtechniques[0]  # Password Guessing
        return subtechniques[3]                     # Credential Stuffing

    # DoS : choisir selon les bytes
    if label == 'DoS':
        bytes_val = features.get('src_bytes', 0)
        if bytes_val > 10000: return subtechniques[1]  # Service Exhaustion
        return subtechniques[0]                         # OS Exhaustion

    return subtechniques[0]

def _get_alert_level(score: int) -> str:
    if score == 0:  return 'INFO'
    if score <= 4:  return 'LOW'
    if score <= 6:  return 'MEDIUM'
    if score <= 8:  return 'HIGH'
    return 'CRITICAL'

if __name__ == '__main__':
    # Test avec features
    result = map_prediction_to_attck(
        'BRUTE_FORCE', 0.94,
        features={'num_failed_logins': 75}
    )
    print(json.dumps(result, indent=2))
'''

with open('../src/attck_mapper.py', 'w', encoding='utf-8') as f:
    f.write(new_mapper)

print("✅ attck_mapper.py mis à jour !")

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '../src/attck_mapper.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERREUR:", result.stderr)